In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path(r"C:\Multimodal_KIRC_Project")
processed_dir = project_dir / "03_processed_data"

counts = pd.read_csv(
    processed_dir / "TCGA_KIRC_raw_gene_count_matrix.csv",
    index_col="Case ID"
)

gene_annotation = pd.read_csv(
    processed_dir / "TCGA_KIRC_gene_annotation.csv"
)

clinical_survival = pd.read_csv(
    processed_dir / "TCGA_KIRC_clinical_survival_harmonized.csv"
)

pfi_eligible = pd.read_csv(
    processed_dir / "TCGA_KIRC_final_PFI_cohort.csv"
)

print("Raw count matrix:", counts.shape)
print("Clinical-survival:", clinical_survival.shape)
print("PFI eligible:", pfi_eligible.shape)

Raw count matrix: (533, 60660)
Clinical-survival: (533, 23)
PFI eligible: (529, 34)


In [2]:
pfi_ids = pfi_eligible["Case ID"].tolist()

counts_pfi = counts.loc[pfi_ids].copy()

clinical_pfi = (
    clinical_survival
    .set_index("Case ID")
    .loc[pfi_ids]
    .copy()
)

print("RNA patients:", counts_pfi.shape[0])
print("Clinical patients:", clinical_pfi.shape[0])

assert counts_pfi.shape[0] == 529
assert clinical_pfi.shape[0] == 529
assert counts_pfi.index.equals(clinical_pfi.index)

print("RNA and clinical patient ordering aligned.")

RNA patients: 529
Clinical patients: 529
RNA and clinical patient ordering aligned.


In [3]:
outcome_pfi = clinical_pfi[
    ["PFI", "PFI.time"]
].copy()

outcome_pfi = outcome_pfi.rename(
    columns={
        "PFI": "event",
        "PFI.time": "time"
    }
)

outcome_pfi["event"] = outcome_pfi["event"].astype(int)

print(outcome_pfi.shape)
print("Events:", outcome_pfi["event"].sum())
print("Censored:", (outcome_pfi["event"] == 0).sum())

assert len(outcome_pfi) == 529
assert outcome_pfi["event"].sum() == 159
assert (outcome_pfi["time"] > 0).all()

display(outcome_pfi.head())

(529, 2)
Events: 159
Censored: 370


,event,time
Case ID,,
TCGA-3Z-A93Z,0,385.0
TCGA-6D-AA2E,0,362.0
TCGA-A3-3306,0,1120.0
TCGA-A3-3307,0,1436.0
TCGA-A3-3308,0,16.0


In [4]:
clinical_features = clinical_pfi[
    [
        "Age at Index",
        "Tumor Grade",
        "AJCC Stage"
    ]
].copy()

display(clinical_features.head())

print("\nMissing values:")
display(clinical_features.isna().sum())

,Age at Index,Tumor Grade,AJCC Stage
Case ID,,,
TCGA-3Z-A93Z,69,G2,Stage I
TCGA-6D-AA2E,68,G2,Stage I
TCGA-A3-3306,67,G3,Stage I
TCGA-A3-3307,66,G3,Stage III
TCGA-A3-3308,77,G2,Stage III



Missing values:


Age at Index     0
Tumor Grade     10
AJCC Stage       3
dtype: int64

In [5]:
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 2026

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

outer_fold = pd.Series(
    index=outcome_pfi.index,
    dtype="int"
)

for fold, (_, test_idx) in enumerate(
    outer_cv.split(
        np.zeros(len(outcome_pfi)),
        outcome_pfi["event"]
    ),
    start=1
):
    outer_fold.iloc[test_idx] = fold

fold_assignment = pd.DataFrame({
    "Case ID": outcome_pfi.index,
    "PFI_event": outcome_pfi["event"].values,
    "PFI_time": outcome_pfi["time"].values,
    "Outer Fold": outer_fold.values
})

display(
    fold_assignment.groupby("Outer Fold")
    .agg(
        Patients=("Case ID", "count"),
        Events=("PFI_event", "sum")
    )
)

,Patients,Events
Outer Fold,,
1.0,106,32
2.0,106,32
3.0,106,32
4.0,106,32
5.0,105,31


In [6]:
fold_assignment.to_csv(
    processed_dir / "TCGA_KIRC_PFI_outer_fold_assignment.csv",
    index=False
)

In [7]:
fold_assignment["Outer Fold"] = (
    fold_assignment["Outer Fold"]
    .astype(int)
)

assert len(fold_assignment) == 529
assert fold_assignment["Case ID"].nunique() == 529
assert set(fold_assignment["Outer Fold"]) == {1, 2, 3, 4, 5}
assert fold_assignment["PFI_event"].sum() == 159

fold_assignment.to_csv(
    processed_dir / "TCGA_KIRC_PFI_outer_fold_assignment.csv",
    index=False
)

display(
    fold_assignment.groupby("Outer Fold")
    .agg(
        Patients=("Case ID", "count"),
        Events=("PFI_event", "sum")
    )
)

,Patients,Events
Outer Fold,,
1,106,32
2,106,32
3,106,32
4,106,32
5,105,31


In [8]:
for col in ["Tumor Grade", "AJCC Stage"]:
    print(f"\n--- {col} ---")
    display(
        clinical_pfi[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="N")
    )


--- Tumor Grade ---


,Tumor Grade,N
0,G2,226
1,G3,203
2,G4,73
3,G1,12
4,NaN,10
5,GX,5



--- AJCC Stage ---


,AJCC Stage,N
0,Stage I,265
1,Stage III,122
2,Stage IV,81
3,Stage II,57
4,NaN,3
5,Stage IB,1


In [9]:
try:
    import sksurv
    print("scikit-survival version:", sksurv.__version__)
except ImportError:
    print("scikit-survival is not installed.")

scikit-survival version: 0.28.0


In [10]:
%pip install scikit-survival

Note: you may need to restart the kernel to use updated packages.


In [11]:
for col in ["Tumor Grade", "AJCC Stage"]:
    print(f"\n--- {col} ---")
    display(
        clinical_pfi[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="N")
    )


--- Tumor Grade ---


,Tumor Grade,N
0,G2,226
1,G3,203
2,G4,73
3,G1,12
4,NaN,10
5,GX,5



--- AJCC Stage ---


,AJCC Stage,N
0,Stage I,265
1,Stage III,122
2,Stage IV,81
3,Stage II,57
4,NaN,3
5,Stage IB,1


In [12]:
try:
    import sksurv
    print("scikit-survival version:", sksurv.__version__)
except ImportError:
    print("scikit-survival is not installed.")

scikit-survival version: 0.28.0


In [13]:
%pip install scikit-survival

Note: you may need to restart the kernel to use updated packages.


In [14]:
clinical_model_data = clinical_pfi[
    [
        "Age at Index",
        "Tumor Grade",
        "AJCC Stage"
    ]
].copy()

clinical_model_data.columns = [
    "age",
    "grade",
    "stage"
]

display(clinical_model_data.head())

print("\nMissing values:")
print(clinical_model_data.isna().sum())

,age,grade,stage
Case ID,,,
TCGA-3Z-A93Z,69,G2,Stage I
TCGA-6D-AA2E,68,G2,Stage I
TCGA-A3-3306,67,G3,Stage I
TCGA-A3-3307,66,G3,Stage III
TCGA-A3-3308,77,G2,Stage III



Missing values:
age       0
grade    10
stage     3
dtype: int64


In [15]:
from sksurv.util import Surv

y_pfi = Surv.from_arrays(
    event=outcome_pfi["event"].astype(bool).values,
    time=outcome_pfi["time"].astype(float).values
)

print(y_pfi[:5])

[(False,  385.) (False,  362.) (False, 1120.) (False, 1436.)
 (False,   16.)]


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxPHSurvivalAnalysis

numeric_features = ["age"]
categorical_features = ["grade", "stage"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [17]:
clinical_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("cox", CoxPHSurvivalAnalysis(alpha=0.01))
])

In [18]:
from sksurv.metrics import concordance_index_censored

clinical_outer_results = []
clinical_predictions = []

X_clinical = clinical_model_data.copy()

for fold in sorted(fold_assignment["Outer Fold"].unique()):

    test_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] == fold,
        "Case ID"
    ].tolist()

    train_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] != fold,
        "Case ID"
    ].tolist()

    X_train = X_clinical.loc[train_ids]
    X_test = X_clinical.loc[test_ids]

    y_train_df = outcome_pfi.loc[train_ids]
    y_test_df = outcome_pfi.loc[test_ids]

    y_train = Surv.from_arrays(
        event=y_train_df["event"].astype(bool).values,
        time=y_train_df["time"].astype(float).values
    )

    y_test = Surv.from_arrays(
        event=y_test_df["event"].astype(bool).values,
        time=y_test_df["time"].astype(float).values
    )

    # Fit preprocessing + Cox only on outer-training patients
    clinical_pipeline.fit(X_train, y_train)

    # Risk score for untouched outer-test patients
    risk_score = clinical_pipeline.predict(X_test)

    c_index = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk_score
    )[0]

    clinical_outer_results.append({
        "Fold": int(fold),
        "Train N": len(train_ids),
        "Test N": len(test_ids),
        "Test Events": int(y_test_df["event"].sum()),
        "C-index": c_index
    })

    clinical_predictions.extend(
        {
            "Case ID": case_id,
            "Fold": int(fold),
            "PFI_event": int(y_test_df.loc[case_id, "event"]),
            "PFI_time": float(y_test_df.loc[case_id, "time"]),
            "Clinical_Risk": float(score)
        }
        for case_id, score in zip(test_ids, risk_score)
    )

clinical_outer_results = pd.DataFrame(clinical_outer_results)
clinical_predictions = pd.DataFrame(clinical_predictions)

display(clinical_outer_results)

C:\Users\kumia\miniforge3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Fold,Train N,Test N,Test Events,C-index
0,1,423,106,32,0.843975
1,2,423,106,32,0.842516
2,3,423,106,32,0.733364
3,4,423,106,32,0.843736
4,5,424,105,31,0.801642


In [19]:
print("Kernel is working")

Kernel is working


In [20]:
print(
    "Mean outer-fold C-index:",
    clinical_outer_results["C-index"].mean()
)

print(
    "SD outer-fold C-index:",
    clinical_outer_results["C-index"].std(ddof=1)
)

Mean outer-fold C-index: 0.8130467214072006
SD outer-fold C-index: 0.04807864729497017


In [21]:
clinical_predictions = (
    clinical_predictions
    .set_index("Case ID")
    .loc[outcome_pfi.index]
    .reset_index()
)

pooled_cindex = concordance_index_censored(
    clinical_predictions["PFI_event"].astype(bool),
    clinical_predictions["PFI_time"],
    clinical_predictions["Clinical_Risk"]
)[0]

print("Pooled out-of-fold clinical C-index:", pooled_cindex)

Pooled out-of-fold clinical C-index: 0.7598292762325026


In [22]:
clinical_outer_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_outer_fold_results.csv",
    index=False
)

clinical_predictions.to_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_out_of_fold_predictions.csv",
    index=False
)

In [23]:
clinical_outer_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_outer_fold_results.csv",
    index=False
)

clinical_predictions.to_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_out_of_fold_predictions.csv",
    index=False
)

clinical_summary = pd.DataFrame({
    "Model": ["Clinical-only Cox PH"],
    "Predictors": ["Age at Index + Tumor Grade + AJCC Stage"],
    "Endpoint": ["PFI"],
    "N": [529],
    "Events": [159],
    "Mean outer-fold C-index": [
        clinical_outer_results["C-index"].mean()
    ],
    "SD outer-fold C-index": [
        clinical_outer_results["C-index"].std(ddof=1)
    ]
})

display(clinical_summary)

clinical_summary.to_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_model_summary.csv",
    index=False
)

,Model,Predictors,Endpoint,N,Events,Mean outer-fold C-index,SD outer-fold C-index
0,Clinical-only Cox PH,Age at Index + Tumor Grade + AJCC Stage,PFI,529,159,0.813047,0.048079


In [24]:
assert len(clinical_outer_results) == 5
assert clinical_predictions["Case ID"].nunique() == 529
assert clinical_predictions["PFI_event"].sum() == 159

print("Clinical benchmark results saved and verified.")

Clinical benchmark results saved and verified.


In [25]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path(r"C:\Multimodal_KIRC_Project")
processed_dir = project_dir / "03_processed_data"

counts = pd.read_csv(
    processed_dir / "TCGA_KIRC_raw_gene_count_matrix.csv",
    index_col="Case ID"
)

gene_annotation = pd.read_csv(
    processed_dir / "TCGA_KIRC_gene_annotation.csv"
)

pfi_eligible = pd.read_csv(
    processed_dir / "TCGA_KIRC_final_PFI_cohort.csv"
)

fold_assignment = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_outer_fold_assignment.csv"
)

clinical_results = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_outer_fold_results.csv"
)

print("Raw counts:", counts.shape)
print("PFI patients:", pfi_eligible.shape[0])
print("Locked folds:", sorted(fold_assignment["Outer Fold"].unique()))

Raw counts: (533, 60660)
PFI patients: 529
Locked folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [26]:
pfi_outcome = (
    pfi_eligible[
        ["Case ID", "PFI", "PFI.time"]
    ]
    .set_index("Case ID")
    .rename(
        columns={
            "PFI": "event",
            "PFI.time": "time"
        }
    )
)

pfi_ids = pfi_outcome.index.tolist()

counts_pfi = counts.loc[pfi_ids].copy()

assert counts_pfi.shape[0] == 529
assert pfi_outcome["event"].sum() == 159
assert counts_pfi.index.equals(pfi_outcome.index)

print("RNA-only modeling cohort aligned.")

RNA-only modeling cohort aligned.


In [27]:
protein_coding_ids = set(
    gene_annotation.loc[
        gene_annotation["gene_type"] == "protein_coding",
        "gene_id"
    ]
)

protein_columns = [
    gene
    for gene in counts_pfi.columns
    if gene in protein_coding_ids
]

counts_protein = counts_pfi[protein_columns].copy()

print("Protein-coding genes available:", counts_protein.shape[1])

Protein-coding genes available: 19962


In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

In [29]:
def preprocess_rna_train_test(
    X_train_counts,
    X_test_counts,
    min_count=10,
    min_fraction=0.20,
    top_k=1000
):
    # ---------------------------------------------
    # 1. Expression-prevalence filtering
    #    learned from TRAINING patients only
    # ---------------------------------------------
    min_patients = int(
        np.ceil(min_fraction * X_train_counts.shape[0])
    )

    keep_genes = (
        (X_train_counts >= min_count).sum(axis=0)
        >= min_patients
    )

    selected_genes = X_train_counts.columns[keep_genes]

    X_train = X_train_counts[selected_genes].copy()
    X_test = X_test_counts[selected_genes].copy()

    # ---------------------------------------------
    # 2. CPM normalization
    # ---------------------------------------------
    train_library = X_train.sum(axis=1)
    test_library = X_test.sum(axis=1)

    X_train_cpm = X_train.div(
        train_library,
        axis=0
    ) * 1_000_000

    X_test_cpm = X_test.div(
        test_library,
        axis=0
    ) * 1_000_000

    # ---------------------------------------------
    # 3. log2(CPM + 1)
    # ---------------------------------------------
    X_train_log = np.log2(X_train_cpm + 1)
    X_test_log = np.log2(X_test_cpm + 1)

    # ---------------------------------------------
    # 4. Variance ranking learned from TRAINING only
    # ---------------------------------------------
    train_variances = X_train_log.var(axis=0)

    top_genes = (
        train_variances
        .sort_values(ascending=False)
        .head(min(top_k, len(train_variances)))
        .index
    )

    X_train_log = X_train_log[top_genes]
    X_test_log = X_test_log[top_genes]

    # ---------------------------------------------
    # 5. Standardization learned from TRAINING only
    # ---------------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_log)
    X_test_scaled = scaler.transform(X_test_log)

    return {
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "genes": list(top_genes),
        "n_prevalence_genes": len(selected_genes),
        "scaler": scaler
    }

In [30]:
l1_ratio_grid = [
    0.1,
    0.5,
    0.9,
    1.0
]

alpha_grid = np.logspace(
    -3,
    1,
    25
)

print("L1 ratios:", l1_ratio_grid)
print("Number of alpha values:", len(alpha_grid))

L1 ratios: [0.1, 0.5, 0.9, 1.0]
Number of alpha values: 25


In [31]:
from sklearn.model_selection import StratifiedKFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import warnings
import numpy as np
import pandas as pd


def tune_coxnet_inner_cv(
    X,
    y_event,
    y_time,
    l1_ratios=(0.1, 0.5, 0.9, 1.0),
    n_splits=5,
    random_state=2026,
    n_alphas=40
):

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    tuning_rows = []

    for l1_ratio in l1_ratios:

        # -------------------------------------------------
        # Build a data-adaptive alpha path
        # using OUTER-TRAINING data only
        # -------------------------------------------------
        y_outer_train = Surv.from_arrays(
            event=y_event.astype(bool),
            time=y_time.astype(float)
        )

        path_model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            n_alphas=n_alphas,
            alpha_min_ratio=0.01,
            max_iter=300000,
            tol=1e-7
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            path_model.fit(
                X,
                y_outer_train
            )

        candidate_alphas = path_model.alphas_

        # -------------------------------------------------
        # Evaluate each alpha using stratified inner CV
        # -------------------------------------------------
        for alpha in candidate_alphas:

            fold_scores = []

            for train_idx, val_idx in inner_cv.split(
                X,
                y_event
            ):

                y_train = Surv.from_arrays(
                    event=y_event[
                        train_idx
                    ].astype(bool),

                    time=y_time[
                        train_idx
                    ].astype(float)
                )

                y_val = Surv.from_arrays(
                    event=y_event[
                        val_idx
                    ].astype(bool),

                    time=y_time[
                        val_idx
                    ].astype(float)
                )

                model = CoxnetSurvivalAnalysis(
                    l1_ratio=l1_ratio,
                    alphas=[alpha],
                    max_iter=300000,
                    tol=1e-7
                )

                try:

                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        model.fit(
                            X[train_idx],
                            y_train
                        )

                    # -----------------------------
                    # Reject all-zero models
                    # -----------------------------
                    coef = np.asarray(
                        model.coef_
                    ).ravel()

                    if np.all(
                        np.abs(coef) < 1e-12
                    ):
                        continue

                    # -----------------------------
                    # Validation risk prediction
                    # -----------------------------
                    risk = model.predict(
                        X[val_idx]
                    )

                    if not np.isfinite(
                        risk
                    ).all():
                        continue

                    cindex = (
                        concordance_index_censored(
                            y_val["event"],
                            y_val["time"],
                            risk
                        )[0]
                    )

                    if np.isfinite(cindex):
                        fold_scores.append(
                            float(cindex)
                        )

                except Exception:
                    continue

            # -------------------------------------------------
            # Require successful fitting in at least 4/5 folds
            # -------------------------------------------------
            if len(fold_scores) >= 4:

                tuning_rows.append({
                    "l1_ratio":
                        float(l1_ratio),

                    "alpha":
                        float(alpha),

                    "valid_inner_folds":
                        len(fold_scores),

                    "mean_inner_cindex":
                        float(
                            np.mean(
                                fold_scores
                            )
                        ),

                    "sd_inner_cindex":
                        float(
                            np.std(
                                fold_scores,
                                ddof=1
                            )
                        )
                })

    # -----------------------------------------------------
    # Convert successful combinations to DataFrame
    # -----------------------------------------------------
    tuning_df = pd.DataFrame(
        tuning_rows
    )

    if tuning_df.empty:
        raise RuntimeError(
            "No valid Coxnet hyperparameter combinations "
            "were found."
        )

    # -----------------------------------------------------
    # Select best mean C-index;
    # use lower SD as tie-breaker
    # -----------------------------------------------------
    tuning_df = (
        tuning_df
        .sort_values(
            [
                "mean_inner_cindex",
                "sd_inner_cindex"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return tuning_df

In [32]:
print(tune_coxnet_inner_cv)

<function tune_coxnet_inner_cv at 0x000002780B48A980>


In [33]:
rna_outer_results = []
rna_predictions = []
rna_selected_genes = []
rna_tuning_results = []

for fold in sorted(
    fold_assignment["Outer Fold"].astype(int).unique()
):

    print(f"\n{'='*55}")
    print(f"Running outer fold {fold}")
    print(f"{'='*55}")

    test_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] == fold,
        "Case ID"
    ].tolist()

    train_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] != fold,
        "Case ID"
    ].tolist()


Running outer fold 1

Running outer fold 2

Running outer fold 3

Running outer fold 4

Running outer fold 5


In [34]:
from sklearn.model_selection import StratifiedKFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

import warnings
import numpy as np
import pandas as pd


def tune_coxnet_inner_cv(
    X,
    y_event,
    y_time,
    l1_ratios=(0.5, 0.9, 1.0),
    n_splits=3,
    random_state=2026,
    n_alphas=12
):

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    tuning_rows = []

    for l1_ratio in l1_ratios:

        # -------------------------------------------------
        # Build data-adaptive alpha path using only
        # the current OUTER-TRAINING data
        # -------------------------------------------------
        y_outer_train = Surv.from_arrays(
            event=y_event.astype(bool),
            time=y_time.astype(float)
        )

        path_model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            n_alphas=n_alphas,
            alpha_min_ratio=0.01,
            max_iter=150000,
            tol=1e-6
        )

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                path_model.fit(
                    X,
                    y_outer_train
                )

        except Exception:
            continue

        candidate_alphas = path_model.alphas_

        # -------------------------------------------------
        # Test each alpha in inner CV
        # -------------------------------------------------
        for alpha in candidate_alphas:

            fold_scores = []

            for train_idx, val_idx in inner_cv.split(
                X,
                y_event
            ):

                y_train = Surv.from_arrays(
                    event=y_event[
                        train_idx
                    ].astype(bool),

                    time=y_time[
                        train_idx
                    ].astype(float)
                )

                y_val = Surv.from_arrays(
                    event=y_event[
                        val_idx
                    ].astype(bool),

                    time=y_time[
                        val_idx
                    ].astype(float)
                )

                model = CoxnetSurvivalAnalysis(
                    l1_ratio=l1_ratio,
                    alphas=[alpha],
                    max_iter=150000,
                    tol=1e-6
                )

                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        model.fit(
                            X[train_idx],
                            y_train
                        )

                    coef = np.asarray(
                        model.coef_
                    ).ravel()

                    # Reject completely zero models
                    if np.all(
                        np.abs(coef) < 1e-12
                    ):
                        continue

                    risk = model.predict(
                        X[val_idx]
                    )

                    # Reject invalid predictions
                    if not np.isfinite(
                        risk
                    ).all():
                        continue

                    cindex = (
                        concordance_index_censored(
                            y_val["event"],
                            y_val["time"],
                            risk
                        )[0]
                    )

                    if np.isfinite(cindex):
                        fold_scores.append(
                            float(cindex)
                        )

                except Exception:
                    continue

            # Require successful fitting in all 3 folds
            if len(fold_scores) >= 3:

                tuning_rows.append({
                    "l1_ratio":
                        float(l1_ratio),

                    "alpha":
                        float(alpha),

                    "valid_inner_folds":
                        len(fold_scores),

                    "mean_inner_cindex":
                        float(
                            np.mean(
                                fold_scores
                            )
                        ),

                    "sd_inner_cindex":
                        float(
                            np.std(
                                fold_scores,
                                ddof=1
                            )
                        )
                })

    tuning_df = pd.DataFrame(
        tuning_rows
    )

    if tuning_df.empty:
        raise RuntimeError(
            "No valid Coxnet hyperparameter combinations "
            "were found."
        )

    tuning_df = (
        tuning_df
        .sort_values(
            [
                "mean_inner_cindex",
                "sd_inner_cindex"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return tuning_df


print("Fast Coxnet tuning function loaded successfully.")

Fast Coxnet tuning function loaded successfully.


In [35]:
rna_outer_results = []
rna_predictions = []
rna_selected_genes = []
rna_tuning_results = []

for fold in sorted(
    fold_assignment["Outer Fold"].astype(int).unique()
):

    print(f"\n{'='*55}")
    print(f"Running outer fold {fold}")
    print(f"{'='*55}")

    # -------------------------------------------------
    # Locked outer training/test patients
    # -------------------------------------------------
    test_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] == fold,
        "Case ID"
    ].tolist()

    train_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] != fold,
        "Case ID"
    ].tolist()

    X_train_counts = (
        counts_protein
        .loc[train_ids]
        .copy()
    )

    X_test_counts = (
        counts_protein
        .loc[test_ids]
        .copy()
    )

    y_train_df = (
        pfi_outcome
        .loc[train_ids]
        .copy()
    )

    y_test_df = (
        pfi_outcome
        .loc[test_ids]
        .copy()
    )

    # -------------------------------------------------
    # TRAINING-ONLY RNA PREPROCESSING
    # -------------------------------------------------
    prep = preprocess_rna_train_test(
        X_train_counts,
        X_test_counts,
        min_count=10,
        min_fraction=0.20,
        top_k=250
    )

    X_train = prep["X_train"]
    X_test = prep["X_test"]

    y_train_event = (
        y_train_df["event"]
        .astype(int)
        .to_numpy()
    )

    y_train_time = (
        y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    print(
        "Genes after prevalence filter:",
        prep["n_prevalence_genes"]
    )

    print(
        "Genes entering Coxnet:",
        X_train.shape[1]
    )

    # -------------------------------------------------
    # INNER CV HYPERPARAMETER TUNING
    # -------------------------------------------------
    tuning = tune_coxnet_inner_cv(
        X_train,
        y_train_event,
        y_train_time,
        l1_ratios=(0.5, 0.9, 1.0),
        n_splits=3,
        random_state=2026 + int(fold),
        n_alphas=12
    )

    best = tuning.iloc[0]

    best_l1 = float(
        best["l1_ratio"]
    )

    best_alpha = float(
        best["alpha"]
    )

    print(
        "Best l1_ratio:",
        best_l1
    )

    print(
        "Best alpha:",
        best_alpha
    )

    print(
        "Best inner C-index:",
        round(
            float(
                best["mean_inner_cindex"]
            ),
            4
        )
    )

    # Save tuning table for audit
    tuning_export = tuning.copy()

    tuning_export[
        "Outer Fold"
    ] = int(fold)

    rna_tuning_results.append(
        tuning_export
    )

    # -------------------------------------------------
    # FINAL FIT ON ENTIRE OUTER-TRAINING SET
    # -------------------------------------------------
    y_train = Surv.from_arrays(
        event=y_train_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    final_model = CoxnetSurvivalAnalysis(
        l1_ratio=best_l1,
        alphas=[best_alpha],
        max_iter=150000,
        tol=1e-6
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        final_model.fit(
            X_train,
            y_train
        )

    # -------------------------------------------------
    # OUTER-TEST PREDICTIONS
    # -------------------------------------------------
    risk_test = final_model.predict(
        X_test
    )

    y_test = Surv.from_arrays(
        event=y_test_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_test_df["time"]
        .astype(float)
        .to_numpy()
    )

    outer_cindex = (
        concordance_index_censored(
            y_test["event"],
            y_test["time"],
            risk_test
        )[0]
    )

    # -------------------------------------------------
    # SELECTED NONZERO GENES
    # -------------------------------------------------
    coefs = np.asarray(
        final_model.coef_
    ).ravel()

    nonzero = (
        np.abs(coefs) > 1e-12
    )

    selected_gene_names = np.asarray(
        prep["genes"]
    )[nonzero]

    print(
        "Selected nonzero genes:",
        int(nonzero.sum())
    )

    print(
        "Outer-test C-index:",
        round(
            float(
                outer_cindex
            ),
            4
        )
    )

    # -------------------------------------------------
    # STORE OUTER-FOLD RESULTS
    # -------------------------------------------------
    rna_outer_results.append({
        "Fold":
            int(fold),

        "Train N":
            len(train_ids),

        "Test N":
            len(test_ids),

        "Test Events":
            int(
                y_test_df[
                    "event"
                ].sum()
            ),

        "Genes after prevalence filter":
            int(
                prep[
                    "n_prevalence_genes"
                ]
            ),

        "Top variable genes":
            int(
                len(
                    prep["genes"]
                )
            ),

        "Selected nonzero genes":
            int(
                nonzero.sum()
            ),

        "Best l1_ratio":
            best_l1,

        "Best alpha":
            best_alpha,

        "Inner CV C-index":
            float(
                best[
                    "mean_inner_cindex"
                ]
            ),

        "Outer C-index":
            float(
                outer_cindex
            )
    })

    # -------------------------------------------------
    # STORE SELECTED GENES
    # -------------------------------------------------
    for gene in selected_gene_names:

        rna_selected_genes.append({
            "Fold":
                int(fold),

            "gene_id":
                gene
        })

    # -------------------------------------------------
    # STORE OUT-OF-FOLD PATIENT PREDICTIONS
    # -------------------------------------------------
    for case_id, score in zip(
        test_ids,
        risk_test
    ):

        rna_predictions.append({
            "Case ID":
                case_id,

            "Fold":
                int(fold),

            "PFI_event":
                int(
                    y_test_df.loc[
                        case_id,
                        "event"
                    ]
                ),

            "PFI_time":
                float(
                    y_test_df.loc[
                        case_id,
                        "time"
                    ]
                ),

            "RNA_Risk":
                float(score)
        })


# =====================================================
# CONVERT RESULTS AFTER ALL FIVE FOLDS FINISH
# =====================================================

rna_outer_results = pd.DataFrame(
    rna_outer_results
)

rna_predictions = pd.DataFrame(
    rna_predictions
)

rna_selected_genes = pd.DataFrame(
    rna_selected_genes
)

rna_tuning_results = pd.concat(
    rna_tuning_results,
    ignore_index=True
)

# -----------------------------------------------------
# SHOW RESULTS
# -----------------------------------------------------
display(
    rna_outer_results
)

rna_mean_cindex = (
    rna_outer_results[
        "Outer C-index"
    ].mean()
)

rna_sd_cindex = (
    rna_outer_results[
        "Outer C-index"
    ].std(ddof=1)
)

print(
    "\nRNA mean outer-fold C-index:",
    rna_mean_cindex
)

print(
    "RNA SD outer-fold C-index:",
    rna_sd_cindex
)

# -----------------------------------------------------
# FINAL INTEGRITY CHECKS
# -----------------------------------------------------
print(
    "\nResults object type:",
    type(
        rna_outer_results
    )
)

print(
    "Results shape:",
    rna_outer_results.shape
)

print(
    "Out-of-fold predictions:",
    len(
        rna_predictions
    )
)

print(
    "Unique predicted patients:",
    rna_predictions[
        "Case ID"
    ].nunique()
)

assert (
    len(
        rna_outer_results
    )
    == 5
)

assert (
    len(
        rna_predictions
    )
    == 529
)

assert (
    rna_predictions[
        "Case ID"
    ].nunique()
    == 529
)

print(
    "\nAll RNA outer-CV integrity checks passed."
)


Running outer fold 1
Genes after prevalence filter: 16193
Genes entering Coxnet: 250
Best l1_ratio: 0.5
Best alpha: 0.10219315773607786
Best inner C-index: 0.7041
Selected nonzero genes: 23
Outer-test C-index: 0.71

Running outer fold 2
Genes after prevalence filter: 16185
Genes entering Coxnet: 250
Best l1_ratio: 0.5
Best alpha: 0.06700192261442559
Best inner C-index: 0.7016
Selected nonzero genes: 42
Outer-test C-index: 0.7605

Running outer fold 3
Genes after prevalence filter: 16175
Genes entering Coxnet: 250
Best l1_ratio: 0.5
Best alpha: 0.044625510850307235
Best inner C-index: 0.724
Selected nonzero genes: 56
Outer-test C-index: 0.7054

Running outer fold 4
Genes after prevalence filter: 16176
Genes entering Coxnet: 250
Best l1_ratio: 0.5
Best alpha: 0.06962628328564303
Best inner C-index: 0.7318
Selected nonzero genes: 34
Outer-test C-index: 0.6451

Running outer fold 5
Genes after prevalence filter: 16193
Genes entering Coxnet: 250
Best l1_ratio: 0.5
Best alpha: 0.04615783316

,Fold,Train N,Test N,Test Events,Genes after prevalence filter,Top variable genes,Selected nonzero genes,Best l1_ratio,Best alpha,Inner CV C-index,Outer C-index
0,1,423,106,32,16193,250,23,0.5,0.102193,0.704130,0.710034
1,2,423,106,32,16185,250,42,0.5,0.067002,0.701569,0.760521
2,3,423,106,32,16175,250,56,0.5,0.044626,0.724014,0.705444
3,4,423,106,32,16176,250,34,0.5,0.069626,0.731798,0.645133
4,5,424,105,31,16193,250,57,0.5,0.046158,0.732828,0.674875



RNA mean outer-fold C-index: 0.6992012672267691
RNA SD outer-fold C-index: 0.043103154855915544

Results object type: <class 'pandas.DataFrame'>
Results shape: (5, 11)
Out-of-fold predictions: 529
Unique predicted patients: 529

All RNA outer-CV integrity checks passed.


In [36]:
rna_outer_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_RNA_outer_fold_results.csv",
    index=False
)

rna_predictions.to_csv(
    processed_dir / "TCGA_KIRC_PFI_RNA_out_of_fold_predictions.csv",
    index=False
)

rna_selected_genes.to_csv(
    processed_dir / "TCGA_KIRC_PFI_RNA_selected_genes.csv",
    index=False
)

rna_tuning_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_RNA_inner_tuning_results.csv",
    index=False
)

In [37]:
clinical_results = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_outer_fold_results.csv"
)

comparison = pd.DataFrame({
    "Fold": clinical_results["Fold"],
    "Clinical C-index": clinical_results["C-index"],
    "RNA C-index": rna_outer_results["Outer C-index"]
})

comparison["RNA - Clinical"] = (
    comparison["RNA C-index"]
    - comparison["Clinical C-index"]
)

display(comparison)

print(
    "Mean Clinical C-index:",
    comparison["Clinical C-index"].mean()
)

print(
    "Mean RNA C-index:",
    comparison["RNA C-index"].mean()
)

print(
    "Mean difference (RNA - Clinical):",
    comparison["RNA - Clinical"].mean()
)

,Fold,Clinical C-index,RNA C-index,RNA - Clinical
0,1,0.843975,0.710034,-0.133941
1,2,0.842516,0.760521,-0.081996
2,3,0.733364,0.705444,-0.027920
3,4,0.843736,0.645133,-0.198603
4,5,0.801642,0.674875,-0.126767


Mean Clinical C-index: 0.8130467214072006
Mean RNA C-index: 0.6992012672267691
Mean difference (RNA - Clinical): -0.11384545418043142
